In [16]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

TRAIN_DIR = PROJECT_ROOT / "student_resource" / "dataset" / "train"

SRC_DIR = (
    PROJECT_ROOT
    / "code"
    / "business_entity_resolution"
    / "src"
)

sys.path.insert(0, str(SRC_DIR))

In [17]:
from business_entity_resolution.preprocessing.normalization import (
    normalize_text,
    normalize_name,
    normalize_address
)

print("Normalization module imported successfully!")

Normalization module imported successfully!


In [18]:
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t"
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)


In [5]:
for df in [s1, s2, s3]:

    df["name_norm"] = df["business_name"].map(
        normalize_name
    )

    df["address_norm"] = df["business_address"].map(
        normalize_address
    )

In [6]:
display(
    s1[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm"
        ]
    ].head(20)
)

,business_name,name_norm,business_address,address_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


In [22]:
gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t"
)

gt["matched_entity_ids"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
)

gt_positive = gt[
    gt["matched_entity_ids"].str.strip() != ""
].copy()

print("Positive S1 entities:", len(gt_positive))

Positive S1 entities: 2083574


In [23]:
sample_gt = gt_positive.sample(
    n=min(10000, len(gt_positive)),
    random_state=42
).copy()

In [24]:
pairs = (
    sample_gt[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x:
            x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

pairs["matched_entity_id"] = (
    pairs["matched_entity_id"]
    .str.strip()
)

print("Positive pairs:", len(pairs))

Positive pairs: 36703


In [25]:
pairs = pairs.merge(
    s1[
        [
            "entity_id",
            "name_norm",
            "address_norm"
        ]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

pairs = pairs.rename(columns={
    "name_norm": "s1_name_norm",
    "address_norm": "s1_address_norm"
})

pairs = pairs.drop(columns=["entity_id"])

In [26]:
pairs_s2 = pairs[
    pairs["matched_entity_id"].str.startswith("S2-")
].copy()

pairs_s2 = pairs_s2.merge(
    s2[
        [
            "entity_id",
            "name_norm",
            "address_norm"
        ]
    ],
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left"
)

pairs_s2 = pairs_s2.rename(columns={
    "name_norm": "matched_name_norm",
    "address_norm": "matched_address_norm"
})

pairs_s2 = pairs_s2.drop(columns=["entity_id"])

In [ ]:
pairs_s3 = pairs[
    pairs["matched_entity_id"].str.startswith("S3-")
].copy()

pairs_s3 = pairs_s3.merge(
    s3[
        [
            "entity_id",
            "name_norm",
            "address_norm"
        ]
    ],
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left"
)

pairs_s3 = pairs_s3.rename(columns={
    "name_norm": "matched_name_norm",
    "address_norm": "matched_address_norm"
})

pairs_s3 = pairs_s3.drop(columns=["entity_id"])

In [ ]:
normalized_pairs = pd.concat(
    [pairs_s2, pairs_s3],
    ignore_index=True
)

print(
    "Normalized positive pairs:",
    len(normalized_pairs)
)

In [ ]:
name_norm_exact = (
    normalized_pairs["s1_name_norm"]
    ==
    normalized_pairs["matched_name_norm"]
)

address_norm_exact = (
    normalized_pairs["s1_address_norm"]
    ==
    normalized_pairs["matched_address_norm"]
)

print(
    "Normalized name exact:",
    round(name_norm_exact.mean() * 100, 2),
    "%"
)

print(
    "Normalized address exact:",
    round(address_norm_exact.mean() * 100, 2),
    "%"
)

In [19]:
examples = [
    "ABC Pvt. Ltd.",
    "ABC Private Limited",
    "Superior Outdoor Corp.",
    "Superior Outdoor Corporation",
    "12 M.G. Road, Pune",
    "12 MG Rd., Pune",
    "100 Main St.",
    "500 Oak Ave."
]

for x in examples:
    print(f"{x:40} -> {normalize_text(x)}")

ABC Pvt. Ltd.                            -> abc pvt ltd
ABC Private Limited                      -> abc private limited
Superior Outdoor Corp.                   -> superior outdoor corp
Superior Outdoor Corporation             -> superior outdoor corporation
12 M.G. Road, Pune                       -> 12 m g road pune
12 MG Rd., Pune                          -> 12 mg rd pune
100 Main St.                             -> 100 main st
500 Oak Ave.                             -> 500 oak ave


In [20]:
from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address
)

print("NAME")
print(normalize_name("ABC Pvt. Ltd."))
print(normalize_name("ABC Private Limited"))
print(normalize_name("Superior Outdoor Corp."))

print("\nADDRESS")
print(normalize_address("12 M.G. Road, Pune"))
print(normalize_address("12 MG Rd., Pune"))
print(normalize_address("500 Oak Ave."))

NAME
abc private limited
abc private limited
superior outdoor corporation

ADDRESS
12 m g road pune
12 mg road pune
500 oak avenue


In [21]:
for df in [s1, s2, s3]:

    df["name_norm"] = df["business_name"].map(
        normalize_name
    )

    df["address_norm"] = df["business_address"].map(
        normalize_address
    )